# Weak-lensing galaxy shape catalogue validation

## Maps

Contents.
- Creat convergence maps

> **_NOTE:_** Before running this notebook, set kernel to `main_set.ipynb'

In [ ]:
from scipy.ndimage.filters import gaussian_filter

from lenspack.utils import bin2d
from lenspack.image.inversion import ks93

In [ ]:
from sp_validation.survey import *
from sp_validation.util import *
from sp_validation.basic import *
from sp_validation.plots import *
from sp_validation.cosmology import *

# Pixelise ellipticities

In [ ]:
# Compute number of pixels
Nx = int(size_x_deg / pixel_size_emap_amin * 60)
Ny = int(size_y_deg / pixel_size_emap_amin * 60)
print_stats(f'Numbers of elipticity pixels for KS93 = ({Nx}, {Ny})', stats_file, verbose=verbose)

In [ ]:
# Bin in 2D
g1_tmp, g2_tmp = bin2d(
    x,
    y,
    npix=(Nx, Ny), 
    v=(g_corr_mc_ngmix[0], g_corr_mc_ngmix[1]),
    extent=(min_x, max_x, min_y, max_y)
)

g_corr_mc_ngmix_map = np.array([g1_tmp, g2_tmp])

# Create convergence maps

In [ ]:
# Transform gamma -> kappa using the Kaiser-Squires (1993) algorithm
kappaE, kappaB = ks93(g1_sign * g_corr_mc_ngmix_map[0], g2_sign * g_corr_mc_ngmix_map[1])

In [ ]:
# Smooth with Gaussian filter
kappaE_sm = gaussian_filter(kappaE, smoothing_scale_pix)
kappaB_sm = gaussian_filter(kappaB, smoothing_scale_pix)

# Get known cluster positions

In [ ]:
# Get cluster information

sp_base = f'{os.environ["HOME"]}/sp_validation'
for sc in ['cosmology']:
    script = os.path.join(sp_base, 'sp_validation', sc)
    %run $script


cluster_cat_name = 'HFI_PCCS_SZ-union_R2.08.fits.gz'
vos_dir = 'vos:cfis/cosmostat/cosmology/external/Planck'

clusters = get_clusters(cluster_cat_name, vos_dir, data_dir, name, verbose=verbose)

print_stats(f"{len(clusters['ra'])} clusters found in {name} footprint", stats_file, verbose=verbose)

In [ ]:
# Project cluster positions
x_cluster, y_cluster =  radec2xy(ra_ngmix_mean, dec_ngmix_mean, clusters['ra'], clusters['dec'])
clusters['x'] = x_cluster
clusters['y'] = y_cluster

# Plot maps

In [ ]:
def get_ticks(loc, N, new_min, new_max):
    """Get ticks
    
    Return formatted axis ticks for plots.
    
    Parameters
    ----------
    loc : array of floats
        original tick locations
    N : number of pixels (in origina coordinates)
    new_min : float
        new coordinate minimum
    new_max : float
        new coordinate maximum
        
    Returns
    -------
    loc_new : array of floats
        new tick locations
    labels_new : array of strings
        new tick labels
    """
    
    loc_new = []
    labels_new = []

    for i in range(1, len(loc)-1):
        lab = loc[i] / N * (new_max - new_min) + new_min
        #print(loc[i], lab)
        loc_new.append(loc[i])
        labels_new.append(f'{lab:.1f}')

    return loc_new, labels_new


def plot_map(m, ra, dec, title, out_path, vlim=None, clusters=None):
    """Plot Map
    
    Plots 2D map.
    
    Parameters
    ----------
    m : 2D array of float
        map
    ra, dec : array of float
        coordinates, for axis ticks
    title : string
        plot title
    out_path : string
        output file path
    vlim : array(2) of float, optional, default=None
        limits of map values, if not given compute from map
    clusters :
        dictionary of cluster information, optional, default=None
    """
    
    plt.figure(figsize=(10, 10))

    # plot image
    plt.imshow(m)

    # save image limits
    xlim = plt.xlim()
    ylim = plt.ylim()

    # Set colorbar
    if not vlim:
       vlim = plt.gci().get_clim()
    else:
        plt.gci().set_clim(vlim)
    plt.colorbar()

    # Transform axis labels to ra, dec
    ra_min, ra_max = ra_ngmix.min(), ra_ngmix.max()
    dec_min, dec_max = dec_ngmix.min(), dec_ngmix.max()

    loc, labels = plt.xticks()
    loc_ra, labels_ra = get_ticks(loc, Nx, ra_min, ra_max)
    plt.xticks(loc_ra, labels=labels_ra)
 
    loc, labels = plt.yticks()
    loc_dec, labels_dec = get_ticks(loc, Ny, dec_min, dec_max)
    plt.yticks(loc_dec, labels=labels_dec)
    
    # plot grid
    grid_lines_ra = []
    grid_lines_dec = []
    n_per_line = 200
 
    # create lines of constant ra and varying dec, and vice versa
    
    # extend beyond projected image limits, to avoid image edges without grid lines 
    d = 2
    gl_ra = np.linspace(ra_min-d, ra_max+d, num=n_per_line)
    gl_dec = np.linspace(dec_min-d, dec_max+d, num=n_per_line)
    ra_list = np.arange(np.floor(ra_min-d), np.ceil(ra_max+d))
    dec_list = np.arange(np.floor(dec_min-d), np.ceil(dec_max+d))
    for ra in ra_list:
        grid_lines_ra.append([ra] * n_per_line)
        grid_lines_dec.append(gl_dec)
    for dec in dec_list:
        grid_lines_dec.append([dec] * n_per_line)
        grid_lines_ra.append(gl_ra)
 
    mean_x = (min_x + max_x) / 2
    mean_y = (min_y + max_y) / 2

    for grid_line_ra, grid_line_dec in zip(grid_lines_ra, grid_lines_dec):
        x, y = radec2xy(ra_ngmix_mean, dec_ngmix_mean, grid_line_ra, grid_line_dec)
        xx = (x + mean_x - min_x) / (max_x - min_x) * Nx
        yy = (y + mean_y - min_y) / (max_y - min_y) * Ny
        plt.plot(xx, yy, 'w:', linewidth=0.5)
    
    # mark cluster positions 
    if clusters:
        x_cluster = (clusters['x'] + mean_x - min_x) / (max_x - min_x) * Nx
        y_cluster = (clusters['y'] + mean_y - min_y) / (max_y - min_y) * Ny
        dy = 0.02
        plt.plot(x_cluster, y_cluster, 'ro', mfc='none', markeredgewidth=0.9, markersize=12)
        
    # go back to image limits
    plt.xlim(xlim)
    plt.ylim(ylim)
 
    plt.gca().invert_yaxis()
    plt.gca().invert_xaxis()
    plt.xlabel('R.A. [deg]')
    plt.ylabel('Dec [deg]')

    plt.title(title)

    plt.savefig(out_path)
    
    return vlim

## Convergence maps

In [ ]:
title = '$\kappa_{\\rm E}$'
out_path = f'{plot_dir}/kappa_E.png'

vlim = plot_map(kappaE_sm, ra_ngmix, dec_ngmix, title, out_path, clusters=clusters)

In [ ]:
title = '$\kappa_{\\rm B}$'
out_path = f'{plot_dir}/kappa_B.png'

plot_map(kappaB_sm, ra_ngmix, dec_ngmix, title, out_path, vlim=vlim, clusters=clusters)

## Stacked convergence maps

In [ ]:
for sc in ['cosmology']:
    script = os.path.join(sp_base, 'sp_validation', sc)
    %run $script


radius = 5

# Stack galaxies
res_stack_mm = stack_mm3(
    ra_ngmix,
    dec_ngmix,
    g_corr_mc_ngmix[0],
    g_corr_mc_ngmix[1],
    w_ngmix,
    clusters['ra'],
    clusters['dec'],
    clusters['z'],
    radius=radius, n_match=1000000
)

In [ ]:
# Plot stacked galaxy density, to check how uniform distribution is. Sometimes at the edges the number
# of galaxies drops visibly

plt.figure(figsize=(10, 10))
plt.hexbin(res_stack_mm[0], res_stack_mm[1], gridsize=100, cmap='gist_stern')
cbar = plt.colorbar()
cbar.set_label('Number count', rotation=270)
plt.title('Density plot')

In [ ]:
# Bin stacked ellipticities

npix = 2048
e1map_stack, e2map_stack = bin2d(
    res_stack_mm[0],
    res_stack_mm[1],
    v=(res_stack_mm[2], -res_stack_mm[3]),
    w=res_stack_mm[4], 
    npix=npix
)

In [ ]:
# transform to gamma -> kappa via the aisers & Squires (1993) algorithm
kappaE_stack, kappaB_stack = ks93(e1map_stack, e2map_stack)

# Smooth
kappaE_stack_sm = gaussian_filter(kappaE_stack, smoothing_scale_pix)
kappaB_stack_sm = gaussian_filter(kappaB_stack, smoothing_scale_pix)

In [ ]:
def plot_map_stacked(kappa, title, output_path, vlim=None):
    """Plot Map Stacked
    
    Plot stacked convergence map.
    
    Parameters
    ----------
    kappa : image
        map values
    title : string
        plot title
    output_path : string
        figure output file path
   
    vlim : array(2) of float, optional, default=None
        map limits; min and max of kappa if not given

    Returns
    -------
    vlim : array(2) of float
        map limits
    """
    
    plt.figure(figsize=(10, 10))

    # plot image
    plt.imshow(kappa)
    
    # set colorbar
    if not vlim:
       vlim = plt.gci().get_clim()
    else:
        plt.gci().set_clim(vlim)
    plt.colorbar()

    npix = kappa.shape[0]
    
    # mark center
    plt.plot(npix/2 - 1, npix/2 - 1, '+')

    # axes ticks
    n_ticks = 4
    loc = np.arange(0, npix + npix / n_ticks, step=npix / n_ticks)
    lab = np.round(
        np.arange(-radius, radius + radius * 2 / n_ticks, step=radius*2/n_ticks),
        1
    )
    plt.xticks(loc, labels=lab)
    plt.yticks(loc, labels=lab)

    plt.xlabel(r'separation $R$ [Mpc]')
    plt.ylabel(r'separation $R$ [Mpc]')
    
    plt.title(title)

    plt.savefig(output_path)
    
    return vlim

In [ ]:
title = 'kappa_E'
output_path = f'{plot_dir}/kappaE_stacked.png'

vlim = plot_map_stacked(kappaE_stack_sm, title, output_path)

In [ ]:
title = 'kappa_B'
output_path = f'{plot_dir}/kappaB_stacked.png'

vlim = plot_map_stacked(kappaB_stack_sm, title, output_path, vlim=vlim)